In [ ]:
!git clone --recursive https://github.com/J-C-Zhao/StructSplat.git
%cd StructSplat

!pip install torch pytorch_lightning huggingface_hub timm deepspeed third_party/vggt gsplat scikit-image lpips transformers ml_collections matplotlib prefetch_generator einops
!pip install --upgrade "numpy>=2.0.0"

Cloning into 'StructSplat'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 206 (delta 60), reused 178 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 2.10 MiB | 19.57 MiB/s, done.
Resolving deltas: 100% (60/60), done.
Submodule 'third_party/vggt' (https://github.com/facebookresearch/vggt.git) registered for path 'third_party/vggt'
Cloning into '/content/StructSplat/third_party/vggt'...
remote: Enumerating objects: 1281, done.        
remote: Counting objects: 100% (8/8), done.        
remote: Compressing objects: 100% (7/7), done.        
remote: Total 1281 (delta 2), reused 1 (delta 1), pack-reused 1273 (from 2)        
Receiving objects: 100% (1281/1281), 64.95 MiB | 41.65 MiB/s, done.
Resolving deltas: 100% (589/589), done.
Submodule path 'third_party/vggt': checked out 'a288dd0f14786c93483e45524328726ab7b1b4ce'
/content/StructSplat
Processing ./third_par

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 49.8 MB/s eta 0:00:0000:01
  Attempting uninstall: timm
    Found existing installation: timm 1.0.30
    Uninstalling timm-1.0.30:
      Successfully uninstalled timm-1.0.30
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 97.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vggt 0.0.1 requires numpy<2, but you have numpy 2.5.4 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.4 which is incompatible.


In [3]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [4]:
!hf download facebook/VGGT-1B --local-dir ckpts/vggt
!hf download facebook/dinov3-convnext-large-pretrain-lvd1689m --local-dir ckpts/dinov3_convnext_large
!hf download DazzlingSun/structsplat pytorch_model.bin --local-dir ckpts/structsplat

Hint: A new version of huggingface_hub (2.2.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/10.1G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/10.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/10.1G [00:00<?, ?B/s]

Fetching 5 files:  20% 1/5 [00:00<00:01,  2.81it/s]
Reconstructing (incomplete total...):   0% 2.06k/10.1G [00:00<475:49:05, 5.87kB/s]
Reconstructing (incomplete total...):  41% 4.09G/10.1G [00:26<00:45, 131MB/s,  167MB/s  ]
Reconstructing (incomplete total...):  44% 4.43G/10.1G [00:30<01:12, 77.4MB/s,  127MB/s  ]
Reconstructing (incomplete total...):  50% 5.06G/10.1G [00:34<00:47, 105MB/s,  193MB/s  ]
Reconstructing (in

In [5]:
from google.colab import drive
drive.mount('/content/drive')
!ls ../drive/MyDrive/Time-Machine/riverview/

Mounted at /content/drive
21_DSC0001.jpg	21_DSC0028.jpg	21_DSC0055.jpg	21_DSC0082.jpg
21_DSC0010.jpg	21_DSC0037.jpg	21_DSC0064.jpg	21_DSC0091.jpg
21_DSC0019.jpg	21_DSC0046.jpg	21_DSC0073.jpg	21_DSC0100.jpg


In [8]:
%cd /content/StructSplat
!sed -i '1i import torch' /content/StructSplat/src/model/semantic_encoder.py
!sed -i 's/AutoModel.from_pretrained(path)/AutoModel.from_pretrained(path, torch_dtype=torch.bfloat16)/g' /content/StructSplat/src/model/semantic_encoder.py

import os
import torch
import gc
from torchvision.transforms import ToTensor
from PIL import Image
from src.config import load_configs
from src.model.gaussian_wrapper import build_gaussian_wrapper

def get_gaussian_wrapper(cfg, ckpt_path):
    print("   -> Loading checkpoint to CPU...")
    param_dict = torch.load(ckpt_path, map_location='cpu')
    
    print("   -> Extracting required weights and converting to bfloat16...")
    module_dict = {"gaussian_predictor": {}}
    for k, v in param_dict.items():
        if k.startswith("gaussian_predictor."):
            module_dict["gaussian_predictor"][k.removeprefix("gaussian_predictor.")] = v.to(torch.bfloat16)
            
    print("   -> Deleting original 5GB checkpoint from System RAM...")
    del param_dict  # Destroy the massive dictionary
    gc.collect()    # Force Python to immediately free up the memory
    
    print("   -> Now building VGGT and DinoV3 safely...")
    wrapper = build_gaussian_wrapper(module_dict, cfg)
    
    for param in wrapper.parameters():
        param.requires_grad = False
    wrapper.eval()
    return wrapper

def main():
    import os
    os.chdir('/content/StructSplat')
    
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    torch.set_float32_matmul_precision('high')

    print("1. Loading config...")
    cfg = load_configs('config/dl3dv.yaml')
    ckpt_path = cfg.gaussian_evaluation_stage.ckpt 
    
    print("2 & 3. Loading models carefully to avoid RAM overflow...")
    # We now pass the path directly so it can manage its own memory!
    model = get_gaussian_wrapper(cfg, ckpt_path)
    
    print("4. Moving assembled model to GPU...")
    model = model.to(torch.bfloat16).to(device)

    target_dir = "../drive/MyDrive/Time-Machine/riverview/"
    image_paths = sorted(os.listdir(target_dir)) 

    print("5. Loading and preprocessing images...")
    images = []
    for path in image_paths:
        img = Image.open(path).convert('RGB')
        img = img.resize((518, 294)) 
        tensor = ToTensor()(img)
        images.append(tensor)
    
    src_tensor = torch.stack(images).unsqueeze(0).to(device).to(torch.bfloat16)
    
    print("6. Running 3D Gaussian prediction (Forward Pass)...")
    with torch.no_grad():
        predictions = model(src_tensor)
        
    print("\n✅ Inference successful!")
    print(f"Predicted {predictions['coordinate'].shape[1]} 3D Gaussians!")

if __name__ == '__main__':
    main()

/content/StructSplat


AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'

In [ ]:
target_dir = "../drive/MyDrive/Time-Machine/riverview/"
image_paths = sorted(os.listdir(target_dir)) 

In [2]:
%cd /content/StructSplat
import json
with open('ckpts/dinov3_convnext_large/config.json', 'r') as f:
    print(json.dumps(json.load(f), indent=2))

/content/StructSplat
{
  "architectures": [
    "DINOv3ConvNextModel"
  ],
  "depths": [
    3,
    3,
    27,
    3
  ],
  "drop_path_rate": 0.0,
  "hidden_act": "gelu",
  "hidden_sizes": [
    192,
    384,
    768,
    1536
  ],
  "image_size": 224,
  "initializer_range": 0.02,
  "layer_norm_eps": 1e-06,
  "layer_scale_init_value": 1e-06,
  "model_type": "dinov3_convnext",
  "num_channels": 3,
  "torch_dtype": "float32",
  "transformers_version": "4.56.0.dev0"
}


In [3]:
!ls ckpts/dinov3_convnext_large

config.json  LICENSE.md  model.safetensors  preprocessor_config.json  README.md


In [2]:
%cd StructSplat

/content/StructSplat


In [4]:
!pip install --upgrade timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 29.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: timm
    Found existing installation: timm 0.9.16
    Uninstalling timm-0.9.16:
      Successfully uninstalled timm-0.9.16
